<a href="https://colab.research.google.com/github/chuy-zip/PROYECTO2_DS/blob/main/PROY2_DS_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!git clone https://github.com/chuy-zip/PROYECTO2_DS.git

fatal: destination path 'PROYECTO2_DS' already exists and is not an empty directory.


In [15]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
import pandas as pd
import numpy as np
import torch.optim as optim
import time
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

In [16]:
df = pd.read_csv("PROYECTO2_DS/data/train_clean.csv")
print(df.head())

print("\n CARGADO DE DATOS ")
df = pd.read_csv("PROYECTO2_DS/data/train_clean.csv")
print(f"Dimensiones del dataset: {df.shape}")

# Ver distribución de clases
print("\n Distribución de clases:")
print(df["discourse_effectiveness"].value_counts())

   Unnamed: 0  discourse_id      essay_id discourse_type  \
0           0  0013cc385424  007ACE74B050           Lead   
1           1  9704a709b505  007ACE74B050       Position   
2           2  c22adee811b6  007ACE74B050          Claim   
3           3  a10d361e54e4  007ACE74B050       Evidence   
4           4  db3e453ec4e2  007ACE74B050   Counterclaim   

  discourse_effectiveness                                         text_clean  
0                Adequate  hi isaac going writing face mar natural landfo...  
1                Adequate  perspective think face natural landform dont t...  
2                Adequate  think face natural landform no life mar descov...  
3                Adequate  life mar would know reason think natural landf...  
4                Adequate  people thought face formed alieans thought lif...  

 CARGADO DE DATOS 
Dimensiones del dataset: (36765, 6)

 Distribución de clases:
discourse_effectiveness
Adequate       20977
Effective       9326
Ineffective     6

In [17]:
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['discourse_effectiveness'])
print(f"\nMapping de clases: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")


Mapping de clases: {'Adequate': np.int64(0), 'Effective': np.int64(1), 'Ineffective': np.int64(2)}


In [18]:
print("\n PREPROCESAMIENTO")

# Tokenizador
def simple_tokenizer(text):
    text = str(text).lower()
    tokens = text.split()
    return tokens

# Aplicar tokenización
df['tokens'] = df['text_clean'].apply(simple_tokenizer)

# Construir vocabulario
def build_vocab(token_lists, min_freq=3, max_vocab_size=30000):
    counter = Counter()
    for tokens in token_lists:
        counter.update(tokens)

    # Ordenar por frecuencia y limitar tamaño
    most_common = counter.most_common(max_vocab_size-2)

    vocab = {'<pad>': 0, '<unk>': 1}
    for idx, (word, count) in enumerate(most_common):
        if count >= min_freq:
            vocab[word] = idx + 2

    print(f"Tamaño del vocabulario: {len(vocab)}")
    return vocab

vocab = build_vocab(df['tokens'])
vocab_size = len(vocab)

# Convertir a índices y aplicar padding
def tokens_to_indices(tokens_list, vocab, max_length=150):
    sequences = []
    for tokens in tokens_list:
        indices = [vocab.get(token, vocab['<unk>']) for token in tokens]
        indices = indices[:max_length]
        sequences.append(torch.tensor(indices, dtype=torch.long))

    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=vocab['<pad>'])
    return padded_sequences

# Convertir textos a tensores
X = tokens_to_indices(df['tokens'], vocab, max_length=150)
y = torch.tensor(df['label'].values, dtype=torch.long)

print(f"Forma de X: {X.shape}")
print(f"Forma de y: {y.shape}")


 PREPROCESAMIENTO
Tamaño del vocabulario: 8457
Forma de X: torch.Size([36765, 150])
Forma de y: torch.Size([36765])


In [19]:
print("\n DIVISIÓN DE DATOS ")

dataset = TensorDataset(X, y)

# 70% train, 15% validation, 15% test
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Entrenamiento: {len(train_dataset)} ejemplos")
print(f"Validación: {len(val_dataset)} ejemplos")
print(f"Prueba: {len(test_dataset)} ejemplos")


 DIVISIÓN DE DATOS 
Entrenamiento: 25735 ejemplos
Validación: 5514 ejemplos
Prueba: 5516 ejemplos


In [20]:
print("\n CONSTRUYENDO MODELO LSTM ")

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, num_classes, dropout=0.3):
        super(LSTMClassifier, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        # LSTM bidireccional para capturar más contexto
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                           batch_first=True, dropout=dropout, bidirectional=True)

        # Capa de atención opcional (mejora performance)
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

        # Capas fully connected
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_dim * 2, 64)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        # Embedding
        embedded = self.embedding(x)  # (batch, seq_len, embedding_dim)

        # LSTM
        lstm_out, (hidden, cell) = self.lstm(embedded)  # (batch, seq_len, hidden_dim*2)

        # Mecanismo de atención simple
        attention_weights = torch.softmax(self.attention(lstm_out).squeeze(-1), dim=1)
        context_vector = torch.sum(attention_weights.unsqueeze(-1) * lstm_out, dim=1)

        # Clasificación
        output = self.dropout(context_vector)
        output = torch.relu(self.fc1(output))
        output = self.dropout(output)
        output = self.fc2(output)

        return output

# Hiperparámetros
EMBEDDING_DIM = 128
HIDDEN_DIM = 64
NUM_LAYERS = 2
NUM_EPOCHS = 10
LEARNING_RATE = 0.001
BATCH_SIZE = 32
NUM_CLASSES = 3

# Instanciar modelo
lstm_model = LSTMClassifier(
    vocab_size=vocab_size,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    num_classes=NUM_CLASSES
)

# Contar parámetros
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Parámetros del modelo: {count_parameters(lstm_model):,}")

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Mover a GPU si está disponible
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lstm_model = lstm_model.to(device)
print(f"Modelo movido a: {device}")


=== CONSTRUYENDO MODELO LSTM ===
Parámetros del modelo: 1,297,924
Modelo movido a: cuda


In [ ]:
print("\n ENTRENANDO MODELO LSTM ")

# Función de entrenamiento mejorada
def train_multiclass_model(model, train_loader, val_loader, num_epochs, learning_rate, num_classes, model_name="LSTM"):
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss()

    train_losses = []
    val_accuracies = []
    best_accuracy = 0

    for epoch in range(num_epochs):
        start_time = time.time()

        # Entrenamiento
        model.train()
        total_loss = 0

        for texts, labels in train_loader:
            texts, labels = texts.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(texts)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        # Validación
        model.eval()
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for texts, labels in val_loader:
                texts, labels = texts.to(device), labels.to(device)
                outputs = model(texts)
                _, predicted = torch.max(outputs, 1)

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        # Métricas
        avg_loss = total_loss / len(train_loader)
        accuracy = accuracy_score(all_labels, all_preds)
        epoch_time = time.time() - start_time

        train_losses.append(avg_loss)
        val_accuracies.append(accuracy)

        # Guardar mejor modelo
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            torch.save(model.state_dict(), 'best_lstm_model.pth')

        print(f'Epoch {epoch+1}/{num_epochs}:')
        print(f'  Loss: {avg_loss:.4f}, Val Accuracy: {accuracy:.4f}, Time: {epoch_time:.2f}s')

        # Reporte de clasificación cada 2 épocas
        if (epoch + 1) % 2 == 0:
            print(f'  Classification Report:')
            print(classification_report(all_labels, all_preds,
                                      target_names=label_encoder.classes_))

    return {
        'train_losses': train_losses,
        'val_accuracies': val_accuracies,
        'best_accuracy': best_accuracy
    }

# Entrenar el modelo
results = train_multiclass_model(
    model=lstm_model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    num_classes=NUM_CLASSES,
    model_name="LSTM"
)

In [ ]:
print("\n=== EVALUACIÓN FINAL ===")

# Cargar mejor modelo
lstm_model.load_state_dict(torch.load('best_lstm_model.pth'))

# Evaluar en test
lstm_model.eval()
test_preds = []
test_labels = []

with torch.no_grad():
    for texts, labels in test_loader:
        texts, labels = texts.to(device), labels.to(device)
        outputs = lstm_model(texts)
        _, predicted = torch.max(outputs, 1)

        test_preds.extend(predicted.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

test_accuracy = accuracy_score(test_labels, test_preds)

print(f"Accuracy en test: {test_accuracy:.4f}")
print("\nReporte de clasificación completo:")
print(classification_report(test_labels, test_preds,
                          target_names=label_encoder.classes_))

# resultados
print("\n" + "="*50)
print("RESULTADOS FINALES LSTM")
print("="*50)
print(f"• Mejor accuracy validación: {results['best_accuracy']:.4f}")
print(f"• Accuracy test: {test_accuracy:.4f}")
print(f"• Parámetros: {count_parameters(lstm_model):,}")
print(f"• Clases: {NUM_CLASSES}")
print(f"• Distribución original: {dict(df['discourse_effectiveness'].value_counts())}")